# 10 — XGBoost (xgboost + Optuna)

**Gradient Boosting con XGBoost.**  
Construye árboles secuencialmente, cada uno corrigiendo los residuos del anterior.

Referencia de clase: `modelos R/árboles/4 - Caso de estudio (XGBoost).R`  
(grid sobre `max.depth` × `eta`, `xgb.cv` con `early_stopping_rounds=50`)

**Estrategia:**
- XGBoost maneja NaN nativamente → **sin imputación**
- `scale_pos_weight = N_neg / N_pos` para el desbalance
- Optuna (20 trials): función objetivo = `xgb.cv` AUC − 0.5·gap (réplica directa del R script)
- `early_stopping_rounds=50` dentro del `xgb.cv` para determinar `n_estimators` óptimo
- Métricas finales sobre train completo + learning curve

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, confusion_matrix, f1_score, roc_curve
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f'XGBoost version: {xgb.__version__}')
print('Imports OK')

In [ ]:
SEED        = 42
N_TRIALS    = 20
EARLY_STOP  = 50   # early stopping rounds — análogo al R script
MAX_ROUNDS  = 1000
np.random.seed(SEED)

BASE_DIR = Path('../../')
DATA_DIR = BASE_DIR / 'data/processed'

matplotlib.rcParams.update({'figure.figsize': (9, 6), 'font.size': 12,
                            'axes.spines.top': False, 'axes.spines.right': False})

In [ ]:
df      = pd.read_parquet(DATA_DIR / 'features_train.parquet')
df_test = pd.read_parquet(DATA_DIR / 'features_test.parquet')

feature_cols = [c for c in df.columns if c not in ('SK_ID_CURR', 'TARGET')]

X           = df[feature_cols].values
y           = df['TARGET'].values
X_test      = df_test[feature_cols].values
sk_ids_test = df_test['SK_ID_CURR'].values

# scale_pos_weight para desbalance de clases
n_neg, n_pos    = (y == 0).sum(), (y == 1).sum()
scale_pos_weight = n_neg / n_pos

print(f'Train: {X.shape}  | Default rate: {y.mean():.2%}')
print(f'NaNs en X: {np.isnan(X).sum():,} (manejados nativamente por XGBoost)')
print(f'scale_pos_weight: {scale_pos_weight:.2f}  (N_neg/N_pos)')

# DMatrix para xgb.cv
dtrain = xgb.DMatrix(X, label=y, feature_names=feature_cols)

In [ ]:
def compute_metrics(y_true, y_prob, threshold=0.5, label='Model'):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    auc  = roc_auc_score(y_true, y_prob)
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return dict(Model=label, AUC=round(auc,4),
                N=len(y_true), P=int(y_pred.sum()),
                TP=int(tp), TN=int(tn), FP=int(fp), FN=int(fn),
                Recall=round(rec,4), Precision=round(prec,4), F1=round(f1,4))

def plot_roc_curve(y_true, y_prob, label, color, ax, title='ROC Curve'):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{label} (AUC = {auc:.4f})')
    ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(title); ax.legend()
    return ax

print('Funciones definidas ✓')

## 1. Optuna — Búsqueda de Hiperparámetros

**Función objetivo** (réplica directa del R script con `xgb.cv` + `early_stopping_rounds`):
$$\text{objetivo} = \text{AUC}_{\text{test-cv}} - 0.5 \cdot \max(0,\ \text{AUC}_{\text{train-cv}} - \text{AUC}_{\text{test-cv}})$$

El número óptimo de rondas de boosting se determina por early stopping dentro del `xgb.cv`.

In [ ]:
best_n_rounds_per_trial = {}  # guardamos las rondas óptimas de cada trial

def objective_xgb(trial):
    params = {
        'max_depth'        : trial.suggest_int('max_depth', 2, 8),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample'        : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight' : trial.suggest_int('min_child_weight', 1, 10),
        'gamma'            : trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        # Fijos
        'objective'        : 'binary:logistic',
        'eval_metric'      : 'auc',
        'scale_pos_weight' : scale_pos_weight,
        'tree_method'      : 'hist',
        'seed'             : SEED,
        'verbosity'        : 0,
    }

    # xgb.cv — idéntico al R script (xgb.cv + early_stopping_rounds)
    cv_results = xgb.cv(
        params              = params,
        dtrain              = dtrain,
        num_boost_round     = MAX_ROUNDS,
        nfold               = 5,
        stratified          = True,
        early_stopping_rounds = EARLY_STOP,
        metrics             = 'auc',
        seed                = SEED,
        as_pandas           = True,
        verbose_eval        = False,
    )

    best_round   = cv_results['test-auc-mean'].idxmax()
    cv_auc       = cv_results['test-auc-mean'].iloc[best_round]
    train_auc    = cv_results['train-auc-mean'].iloc[best_round]

    best_n_rounds_per_trial[trial.number] = best_round + 1  # rounds es 1-indexed

    gap = max(0.0, train_auc - cv_auc)
    return cv_auc - 0.5 * gap

print(f'Lanzando Optuna — {N_TRIALS} trials (XGBoost + xgb.cv + early stopping)...')
study_xgb = optuna.create_study(
    direction = 'maximize',
    sampler   = optuna.samplers.TPESampler(seed=SEED)
)
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS, show_progress_bar=True)

best_n_rounds = best_n_rounds_per_trial[study_xgb.best_trial.number]

print(f'\nMejores hiperparámetros:')
for k, v in study_xgb.best_params.items():
    print(f'  {k:<22}: {v}')
print(f'  {"n_estimators":<22}: {best_n_rounds} (via early stopping)')
print(f'\nObjetivo (CV AUC penalizado): {study_xgb.best_value:.4f}')

In [ ]:
# Historial de Optuna
trial_df = study_xgb.trials_dataframe()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(trial_df['number'], trial_df['value'], 'o-', color='#e74c3c', ms=6)
ax.axhline(study_xgb.best_value, color='#3498db', ls='--',
           label=f'Best = {study_xgb.best_value:.4f}')
ax.set_xlabel('Trial'); ax.set_ylabel('Objetivo (CV AUC penalizado)')
ax.set_title('Optuna — XGBoost')
ax.legend()
plt.tight_layout()
plt.show()


## 2. Modelo Final — Refit en Train Completo

In [ ]:
best_p = study_xgb.best_params

params_final = {
    'max_depth'        : best_p['max_depth'],
    'learning_rate'    : best_p['learning_rate'],
    'subsample'        : best_p['subsample'],
    'colsample_bytree' : best_p['colsample_bytree'],
    'min_child_weight' : best_p['min_child_weight'],
    'gamma'            : best_p['gamma'],
    'reg_alpha'        : best_p['reg_alpha'],
    'reg_lambda'       : best_p['reg_lambda'],
    'objective'        : 'binary:logistic',
    'eval_metric'      : 'auc',
    'scale_pos_weight' : scale_pos_weight,
    'tree_method'      : 'hist',
    'seed'             : SEED,
    'verbosity'        : 1,
}

print(f'Entrenando modelo final ({best_n_rounds} rondas de boosting)...')
evals_result = {}
xgb_final = xgb.train(
    params          = params_final,
    dtrain          = dtrain,
    num_boost_round = best_n_rounds,
    evals           = [(dtrain, 'train')],
    evals_result    = evals_result,
    verbose_eval    = 50,
)

print(f'Modelo final entrenado ✓')

## 3. Métricas sobre Train Completo

In [ ]:
y_prob_train = xgb_final.predict(dtrain)
metrics = compute_metrics(y, y_prob_train, threshold=0.5, label='XGBoost')
metrics['CV_AUC']    = round(study_xgb.best_value, 4)
metrics['n_rounds']  = best_n_rounds

print('=' * 60)
print('XGBOOST — MÉTRICAS FINALES')
print('=' * 60)
for k, v in metrics.items():
    if k != 'Model':
        print(f'  {k:<12}: {v}')
print('=' * 60)

display(pd.DataFrame([metrics]).set_index('Model'))

## 4. Curva ROC

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
plot_roc_curve(y, y_prob_train, label='XGBoost (train)', color='#e74c3c',
               ax=ax, title='ROC — XGBoost')
plt.tight_layout()
plt.show()


## 5. Learning Curve (Train AUC vs Ronda)

Réplica del gráfico de convergencia del R script de XGBoost.

In [ ]:
train_auc_history = evals_result['train']['auc']
rounds = np.arange(1, len(train_auc_history) + 1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(rounds, train_auc_history, color='#e74c3c', lw=2, label='Train AUC')
ax.axhline(train_auc_history[-1], color='#3498db', ls='--', alpha=0.7,
           label=f'AUC final = {train_auc_history[-1]:.4f}')
ax.set_xlabel('Ronda de Boosting'); ax.set_ylabel('AUC')
ax.set_title('Learning Curve — XGBoost')
ax.legend()
plt.tight_layout()
plt.show()


## 6. Importancia de Variables

In [ ]:
# Importancia por gain (equivalente a la importancia del R script)
importance_dict = xgb_final.get_score(importance_type='gain')
feat_imp_df = pd.DataFrame(list(importance_dict.items()),
                            columns=['Feature', 'Gain'])
feat_imp_df = feat_imp_df.sort_values('Gain', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 7))
bars = ax.barh(feat_imp_df['Feature'][::-1], feat_imp_df['Gain'][::-1],
               color='#e74c3c', edgecolor='white')
ax.set_xlabel('Importancia (Gain)')
ax.set_title('Top 15 Features — XGBoost')

for bar, val in zip(bars, feat_imp_df['Gain'][::-1]):
    ax.text(bar.get_width() + max(feat_imp_df['Gain'])*0.01,
            bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()
display(feat_imp_df.reset_index(drop=True))

In [ ]:
# Distribución de scores (TARGET 0 vs 1)
from scipy.stats import gaussian_kde

fig, ax = plt.subplots(figsize=(9, 5))
for val, color, label in [(0, '#3498db', 'TARGET=0 (paga)'),
                           (1, '#e74c3c', 'TARGET=1 (default)')]:
    probs = y_prob_train[y == val]
    kde = gaussian_kde(probs, bw_method=0.1)
    xs  = np.linspace(0, 1, 300)
    ax.plot(xs, kde(xs), color=color, lw=2, label=label)
    ax.fill_between(xs, kde(xs), alpha=0.15, color=color)

ax.set_xlabel('Score predicho P(default)')
ax.set_ylabel('Densidad')
ax.set_title('Distribución de scores — XGBoost')
ax.legend()
plt.tight_layout()
plt.show()


## 7. Predicciones Test

In [ ]:
dtest       = xgb.DMatrix(X_test, feature_names=feature_cols)
y_test_prob = xgb_final.predict(dtest)

submission = pd.DataFrame({'SK_ID_CURR': sk_ids_test, 'TARGET': y_test_prob})
out_path   = DATA_DIR / 'submission_10_xgboost.csv'
submission.to_csv(out_path, index=False)

print(f'Predicciones guardadas: {out_path}')
print(f'Shape: {submission.shape}')
print(f'Score medio predicho: {y_test_prob.mean():.4f}')
display(submission.head())

## Resumen Final

In [ ]:
print('=' * 65)
print('XGBOOST — HIPERPARÁMETROS ÓPTIMOS Y MÉTRICAS')
print('=' * 65)
print('Hiperparámetros:')
for k, v in best_p.items():
    print(f'  {k:<22}: {v}')
print(f'  {"n_estimators":<22}: {best_n_rounds} (early stopping)')
print('\nMétricas finales:')
for k, v in metrics.items():
    if k != 'Model':
        print(f'  {k:<12}: {v}')
print('=' * 65)

## Kaggle Submission — AUC Test (Public Leaderboard)

Envía el CSV y recupera el AUC en el Public LB (~30% del test set).
Polling cada 30 s hasta que el status sea `complete` (máx 3 min).

> **Límite**: 5 submissions/día.


## Kaggle Submission — AUC Test (Public Leaderboard)

Envía el CSV a `home-credit-default-risk` y recupera el AUC del Public LB (~30% del test).
Usa la Python API (`KaggleApi`) directamente — polling cada 10 s, máx 5 min.

> **Límite**: 5 submissions/día.


In [ ]:
from kaggle import KaggleApi
import time

COMPETITION = 'home-credit-default-risk'
_sub_path   = DATA_DIR / 'submission_10_xgboost.csv'
_msg = f"10_xgboost | train={metrics["AUC"]:.4f} | CV={study_xgb.best_value:.4f} | rounds={best_n_rounds}"

def _as_str(x):
    return '' if x is None else str(x)

def _get_score(api, comp, file_name, message, poll=10, timeout=300):
    """Poll until public_score appears on the matching submission."""
    start = time.time()
    while time.time() - start < timeout:
        subs = api.competition_submissions(comp)
        matched = next(
            (s for s in subs
             if _as_str(getattr(s, 'file_name', None)) == file_name
             and _as_str(getattr(s, 'description', None)) == message),
            subs[0] if subs else None
        )
        if matched:
            pub     = _as_str(getattr(matched, 'public_score',  None))
            status  = _as_str(getattr(matched, 'status',        None))
            elapsed = int(time.time() - start)
            print(f'  [{elapsed:>3}s] status={status!r}  public_score={pub!r}')
            if pub and pub.lower() not in ('', 'none', 'null', '-'):
                priv = _as_str(getattr(matched, 'private_score', None))
                return float(pub), (float(priv) if priv and priv.lower() not in ('','none','null','-') else None)
        time.sleep(poll)
    return None, None

_api = KaggleApi()
_api.authenticate()

print(f'Enviando: {_msg}')
_api.competition_submit(file_name=str(_sub_path), message=_msg, competition=COMPETITION)
print('Esperando scoring (poll 10 s / máx 5 min)...')

public_auc, private_auc = _get_score(_api, COMPETITION, _sub_path.name, _msg)
print(f'\nAUC test  Public  LB : {public_auc}')
print(f'AUC test  Private LB : {private_auc}')
